In [1]:
import json
import duckdb

# Pfade zu deinen Dateien
json_path = r"C:\Users\up19040\Projekte\LMU\Programmieren mit LMU\information-extraction-pilot\data\docs\pdf_info.json"
#db_path = r"C:\Users\up19040\Projekte\LMU\Programmieren mit LMU\information-extraction-pilot\data\processed\embeddings\text-embedding-3-large_from_2025_03_06.duckdb"
db_path = r"C:\Users\up19040\Projekte\LMU\Programmieren mit LMU\information-extraction-pilot\data\processed\embeddings\text-embedding-ada-002_N1_from_2025_07_17.duckdb"

# Lade das JSON
with open(json_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

# Alle Dateinamen extrahieren, bei denen "in_sample" sample_160 enthält
#sample_value = "sample_39"
sample_value = "sample_anpassungsplaene_N1"
matching_files = [
    file_name.split('/')[-1]  # Nur den Dateinamen extrahieren
    for file_name, entry in data.items()
    if sample_value in entry.get("in_sample", [])
]

# Verbindung zu DuckDB herstellen
con = duckdb.connect(db_path)

# Zähler für enthaltene und nicht enthaltene PDFs
count_in_db = 0
count_not_in_db = 0

# Für jeden Dateinamen prüfen, ob er schon in der Datenbank ist
for short_file_name in matching_files:
    result = con.execute(
        "SELECT COUNT(*) FROM pdf_files WHERE short_file_name = ?",  
        [short_file_name]
    ).fetchone()[0]
    pdf_in_database = bool(result)
    if pdf_in_database:
        count_in_db += 1
    else:
        count_not_in_db += 1
    #print(f"{short_file_name}: {'Bereits in DB' if pdf_in_database else 'Nicht in DB'}")

con.close()

# Gesamtsummen ausgeben
print(f"\nSumme der enthaltenen PDFs: {count_in_db}")
print(f"Summe der nicht enthaltenen PDFs: {count_not_in_db}")




Summe der enthaltenen PDFs: 91
Summe der nicht enthaltenen PDFs: 0


In [2]:
import duckdb
# Look at one particular embedded pdf
short_file_name = "Anpassungsplan_Stadt_Karlsruhe.pdf"
db_path = r"C:\Users\up19040\Projekte\LMU\Programmieren mit LMU\information-extraction-pilot\data\processed\embeddings\text-embedding-ada-002_N1_from_2025_07_17.duckdb"

con = duckdb.connect(db_path)
# Alle Einträge mit dem gewünschten short_file_name anzeigen
df = con.execute("""
    SELECT *
    FROM pages
    WHERE short_file_name = ?""",  
    [short_file_name]  
).fetchdf()

print(df)

con.close()

                                                    id  \
0    237b51512785d5ebef298db23994c2bdd0690720192137...   
1    a1991b35cf1652ad4b93194688a9cb1696e6a81be29ee7...   
2    928bdbf58a6ed8ad55054edac8c7995bc693e6d8f51d98...   
3    f215c7bf4bb8c659cbb39ced294f5a18068320f2e0a83e...   
4    7182b62bc12a30f567bddf95f7957b35cbc7ac9945c171...   
..                                                 ...   
183  ec53a5443aef99bb3732f47ebaf8831a59e39377091994...   
184  9867f8661e734aa4b070c6aeea58bfdcd2bb04d81ccbf9...   
185  29e45994ddf30f3c7f25d9b75dc1bcaf0cbe8b26eaca3a...   
186  f00133c5fba50ea1aeb8d84496fd7b301953df8c432bae...   
187  59db8e0910f495a2214c85afb7eecefc37492b479d0003...   

                        short_file_name  page_index page_label  \
0    Anpassungsplan_Stadt_Karlsruhe.pdf           0          1   
1    Anpassungsplan_Stadt_Karlsruhe.pdf           1          2   
2    Anpassungsplan_Stadt_Karlsruhe.pdf           2          3   
3    Anpassungsplan_Stadt_Karlsruhe.pdf

In [1]:
import duckdb
# Delete one PDF from the database
short_file_name = "sato holdings_2022_report.pdf"
db_path = r"C:\Users\up19040\Projekte\LMU\Programmieren mit LMU\information-extraction-pilot\data\processed\embeddings\text-embedding-ada-002_from_2025_03_06.duckdb"

con = duckdb.connect(db_path)

con.execute("DELETE FROM pdf_files WHERE short_file_name = ?", [short_file_name])  
con.execute("DELETE FROM pages WHERE short_file_name = ?", [short_file_name])  
con.execute("""PRAGMA create_fts_index('pages', 'id', 'page_content', overwrite=1);""") 

con.close()


In [10]:
import json
import os
from PyPDF2 import PdfReader
import duckdb

# Delete duckdb entries, that are not fully embedded in the database

# Define file paths
json_path = r"C:\Users\up19040\Projekte\LMU\Programmieren mit LMU\information-extraction-pilot\data\docs\pdf_info.json"
db_path = r"C:\Users\up19040\Projekte\LMU\Programmieren mit LMU\information-extraction-pilot\data\processed\embeddings\text-embedding-ada-002_from_2025_03_06.duckdb"
pdf_base_dir = r"C:\Users\up19040\Projekte\LMU\Programmieren mit LMU\information-extraction-pilot\data\pdfs"

# Load JSON data
with open(json_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

# Extract matching files
sample_value = "sample_20250604"
matching_files = [
    file_name.split('/')[-1]
    for file_name, entry in data.items()
    if sample_value in entry.get("in_sample", [])
]

def check_pdf_vs_db(matching_files, db_path, pdf_base_dir, delete_pdfs=False):
    con = duckdb.connect(db_path)
    mismatches = []
    embedded_files = 0
    
    for short_file_name in matching_files:
        # Check if file exists in pdf_files table
        result = con.execute(
            "SELECT COUNT(*) FROM pdf_files WHERE short_file_name = ?",  
            [short_file_name]
        ).fetchone()[0]
        pdf_in_database = bool(result)
        
        # Skip comparison if PDF not in database
        if not pdf_in_database:
            continue
        
        pdf_path = os.path.join(pdf_base_dir, short_file_name)
        
        # Get PDF page count
        try:
            reader = PdfReader(pdf_path)
            num_pdf_pages = len(reader.pages)
        except Exception as e:
            mismatches.append((short_file_name, f"PDF read error: {str(e)}"))
            continue
        
        # Get database row count
        try:
            row_count_df = con.execute("""
                SELECT COUNT(*) AS row_count
                FROM pages
                WHERE short_file_name = ?""",  
                [short_file_name]
            ).fetchdf()
            num_db_rows = row_count_df['row_count'].iloc[0]
        except Exception as e:
            mismatches.append((short_file_name, f"DB query error: {str(e)}"))
            continue
        
        # Compare counts
        if num_pdf_pages - 5 > num_db_rows:
            if delete_pdfs:
                con.execute("DELETE FROM pdf_files WHERE short_file_name = ?", [short_file_name])  
                con.execute("DELETE FROM pages WHERE short_file_name = ?", [short_file_name])  
                con.execute("""PRAGMA create_fts_index('pages', 'id', 'page_content', overwrite=1);""") 
            print(f"Starke Abweichung: {short_file_name} has {num_pdf_pages} pages in PDF but only {num_db_rows} rows in DB. Entry in DB deleted.")
            mismatches.append((short_file_name, f"Pages: {num_pdf_pages} | DB rows: {num_db_rows}"))
        elif num_pdf_pages == num_db_rows:
            embedded_files += 1
        else:   
            #print(f"Leichte Abweichung: {short_file_name} has {num_pdf_pages} pages in PDF but {num_db_rows} rows in DB.")
            mismatches.append((short_file_name, f"Pages: {num_pdf_pages} | DB rows: {num_db_rows}"))
            embedded_files += 1
            
    print(f"Anzahl der vollständig eingebetteten PDFs: {embedded_files}")    
    con.close()
    return mismatches

# Execute and print mismatches
mismatches = check_pdf_vs_db(matching_files, db_path, pdf_base_dir, delete_pdfs=False)

print("\nMismatch Report (only for files in DB):")
if mismatches:
    for file_name, message in mismatches:
        print(f"- {file_name}: {message}")
else:
    print("All PDF page counts match their database row counts for files in DB!")


ignore '/Perms' verify failed


Starke Abweichung: Bus Eireann_2023.pdf has 136 pages in PDF but only 127 rows in DB. Entry in DB deleted.


ignore '/Perms' verify failed
ignore '/Perms' verify failed
ignore '/Perms' verify failed
ignore '/Perms' verify failed
ignore '/Perms' verify failed


Starke Abweichung: unipol_assicurazioni_spa_2023_sustainability_report.pdf has 372 pages in PDF but only 364 rows in DB. Entry in DB deleted.
Anzahl der vollständig eingebetteten PDFs: 842

Mismatch Report (only for files in DB):
- adidas_ag_2023_sustainability_report.pdf: Pages: 352 | DB rows: 349
- akzo_nobel_n.v._2023_sustainability_report.pdf: Pages: 202 | DB rows: 201
- Alpha Bank SA_2024.pdf: Pages: 667 | DB rows: 666
- Atos_2024.pdf: Pages: 478 | DB rows: 476
- british_american_tobacco_plc_2023_sustainability_report.pdf: Pages: 44 | DB rows: 43
- Bus Eireann_2023.pdf: Pages: 136 | DB rows: 127
- buzzi_spa_2023_sustainability_report.pdf: Pages: 133 | DB rows: 131
- carnival_plc_2023_sustainability_report.pdf: Pages: 96 | DB rows: 95
- Cellnex_2024.pdf: Pages: 569 | DB rows: 568
- Commerzbank AG_2024.pdf: Pages: 548 | DB rows: 547
- Enagas SA_2024.pdf: Pages: 370 | DB rows: 369
- Ferrari NV_2024.pdf: Pages: 462 | DB rows: 461
- heineken_2023_sustainability_report.pdf: Pages: 217 |

In [ ]:
# look at database in general
import duckdb

db_path = r"C:\Workspace\BCE\up19040\gist-lmu-bbk\data\processed\embeddings\text-embedding-3-large_m13_from_2025_09_09.duckdb"
con = duckdb.connect(db_path)

# Zeige alle Tabellen in der Datenbank an
tables = con.execute("SHOW TABLES").fetchdf()
print(tables)

con.close()

In [ ]:
# look at each table in database
import duckdb

db_path = r"C:\Workspace\BCE\up19040\gist-lmu-bbk\data\processed\embeddings\text-embedding-3-large_m13_ifrs_from_2025_09_09-part3.duckdb"

con = duckdb.connect(db_path)

schema_pages = con.execute("DESCRIBE pages").fetchdf()
print(schema_pages)
data_pages = con.execute("SELECT * FROM pages").fetchdf()
# print(data_pages)

schema_files = con.execute("DESCRIBE pdf_files").fetchdf()
print(schema_files)
data_files = con.execute("SELECT * FROM pdf_files").fetchdf()
print(data_files)

schema_query = con.execute("DESCRIBE search_query_embeddings").fetchdf()
print(schema_query)
data_query = con.execute("SELECT * FROM search_query_embeddings LIMIT 10").fetchdf()
# print(data_query)


con.close()


In [ ]:
# Hilfsfunktion zum erstellen einer Datenbank wenn sie in mehrere aufteilt wird (nächste Zelle)

import duckdb


def create_database(database_name: str, embed_dim: int = 3072) -> None:
    """Erstellt die Tabellen mit den passenden Datentypen, falls noch nicht vorhanden."""
    with duckdb.connect(database_name) as con:
        con.execute(f"""
        CREATE TABLE IF NOT EXISTS pages(
            id STRING,
            short_file_name STRING,
            page_index INTEGER,
            page_label STRING,
            page_content STRING,
            embedding FLOAT[{embed_dim}],
            embed_created_by STRING,
            embed_created_at STRING
        )
        """)
        con.execute("""
        CREATE TABLE IF NOT EXISTS pdf_files(
            short_file_name STRING,
            complete_file_path STRING,
            number_of_pages INTEGER
        )
        """)
        con.execute(f"""
        CREATE TABLE IF NOT EXISTS search_query_embeddings(
            search_query STRING,
            embedding FLOAT[{embed_dim}],
            embed_created_by STRING,
            embed_created_at STRING
        )
        """)
        # Optional: FTS Extension laden (funktioniert in DuckDB ab Version 0.8.0)
        con.execute("INSTALL fts")
        con.execute("LOAD fts")


In [ ]:
# Datenbank in mehrere aufteilen
import json

import duckdb

# Pfade zur Original- und zum neuen Datenbankfile
original_db_path = r"C:\Workspace\BCE\up19040\gist-lmu-bbk\data\processed\embeddings\text-embedding-3-large_m13_from_2025_09_09.duckdb"
new_db_path = r"C:\Workspace\BCE\up19040\gist-lmu-bbk\data\processed\embeddings\text-embedding-3-large_m13_IFRS_from_2025_09_09-part3.duckdb"

# Lade pdf_info.json und definiere dein gewünschtes Sample
json_path = r"C:\Workspace\BCE\up19040\gist-lmu-bbk\data\docs\pdf_info.json"
sample_name = "sample_M13_IFRS_20250909-part3"

with open(json_path) as f:
    pdf_info = json.load(f)

pdfs_in_sample = [
    pdf_name for pdf_name, info in pdf_info.items() if sample_name in info.get("in_sample", [])
]

# Liste nur mit Dateinamen ohne Pfad
pdfs_in_sample_filenames = [os.path.basename(p) for p in pdfs_in_sample]
print(len(pdfs_in_sample_filenames))
# Erstelle Verbindung zur alten DB und lese alle Tabellendaten
con_old = duckdb.connect(original_db_path)
pages_df = con_old.execute("SELECT * FROM pages").df()
pdf_files_df = con_old.execute("SELECT * FROM pdf_files").df()
search_query_embeddings_df = con_old.execute("SELECT * FROM search_query_embeddings").df()
con_old.close()

# Filter für pages und pdf_files nur mit short_file_name in pdfs_in_sample
pages_df_filtered = pages_df[pages_df["short_file_name"].isin(pdfs_in_sample_filenames)]
pdf_files_df_filtered = pdf_files_df[
    pdf_files_df["short_file_name"].isin(pdfs_in_sample_filenames)
]
# search_query_embeddings bleibt unverändert oder anpassen je nach Bedarf

# Neue DB erstellen (alte evtl. löschen oder überschreiben)
create_database(new_db_path, embed_dim=3072)
con_new = duckdb.connect(new_db_path)
con_new.register("pages_temp", pages_df_filtered)
con_new.execute("INSERT INTO pages SELECT * FROM pages_temp")

con_new.register("pdf_files_temp", pdf_files_df_filtered)
con_new.execute("INSERT INTO pdf_files SELECT * FROM pdf_files_temp")

con_new.register("search_query_embeddings_temp", search_query_embeddings_df)
con_new.execute("INSERT INTO search_query_embeddings SELECT * FROM search_query_embeddings_temp")

con_new.close()

# add fts extension
with duckdb.connect(new_db_path) as con:
    con.execute("""PRAGMA create_fts_index('pages', 'id', 'page_content', overwrite=1);""")
